# Label Timing Uncertainty Audit

Ce notebook reprend le script `label_timing_uncertainty_audit.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Teste la sensibilite aux timestamps d'entree, critique pour une alerte live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Evaluate final predictions under danger-event timestamp uncertainty.
- Commande de reproduction referencee : label timing uncertainty.
- Artefacts controles : Danger-event timestamp uncertainty audit exists. (`runs/exp_038_label_timing_uncertainty/metrics/label_timing_uncertainty_summary.csv`).
- Run par defaut : `runs/exp_038_label_timing_uncertainty`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "label_timing_uncertainty_audit.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_fscore_support, roc_auc_score

from ml_pipeline import ROOT, alarm_episodes, safe_auc, write_json
from sequence_experiments import append_report, make_run_dir


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    return path


## Fonction `load_prediction_files`

Cette cellule definit `load_prediction_files`. Elle prepare une partie du script.

In [ ]:
def load_prediction_files(ensemble_run):
    files = sorted((ensemble_run / "features").glob("ensemble_predictions_seed*.csv"))
    rows = []
    for path in files:
        match = re.search(r"seed(\d+)", path.stem)
        if match:
            rows.append((int(match.group(1)), path))
    return rows


## Fonction `recompute_labels`

Cette cellule definit `recompute_labels`. Elle prepare une partie du script.

In [ ]:
def recompute_labels(df, horizon_s, target_shift_s=0.0, boundary_slack_s=0.0):
    labels = np.zeros(len(df), dtype=np.int32)
    ttt = np.full(len(df), np.nan, dtype=np.float32)
    danger = df["is_danger_clip"].astype(int).to_numpy() == 1
    target = pd.to_numeric(df["target_time_s"], errors="coerce").to_numpy(dtype=np.float32)
    time = df["time_s"].astype(float).to_numpy(dtype=np.float32)
    valid = danger & np.isfinite(target)
    adjusted_target = target + float(target_shift_s)
    ttt[valid] = adjusted_target[valid] - time[valid]
    labels[valid] = ((ttt[valid] >= -float(boundary_slack_s)) & (ttt[valid] <= float(horizon_s) + float(boundary_slack_s))).astype(np.int32)
    return labels, ttt


## Fonction `evaluate_windows`

Cette cellule definit `evaluate_windows`. Elle prepare une partie du script.

In [ ]:
def evaluate_windows(df, score_col, labels):
    scores = df[score_col].to_numpy(dtype=np.float32)
    pred = (scores >= 0.5).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, pred, average="binary", zero_division=0)
    return {
        "positive_windows": int(labels.sum()),
        "window_ap": safe_auc(average_precision_score, labels, scores),
        "window_roc_auc": safe_auc(roc_auc_score, labels, scores),
        "window_precision_at_0_5": float(precision),
        "window_recall_at_0_5": float(recall),
        "window_f1_at_0_5": float(f1),
    }


## Fonction `evaluate_operating_threshold`

Cette cellule definit `evaluate_operating_threshold`. Elle prepare une partie du script.

In [ ]:
def evaluate_operating_threshold(df, score_col, threshold, target_shift_s=0.0, late_grace_s=0.0):
    danger_clips = df[df["is_danger_clip"] == 1]["video_id"].unique().tolist()
    pre_entry_hits = 0
    grace_hits = 0
    missed = 0
    early_times = []
    grace_early_times = []
    for video_id in danger_clips:
        group = df[df["video_id"] == video_id].sort_values("time_s")
        target = pd.to_numeric(group["target_time_s"], errors="coerce").dropna()
        if target.empty:
            continue
        target_time = float(target.iloc[0]) + float(target_shift_s)
        alarms = alarm_episodes(group["time_s"], group[score_col], threshold, gap_s=1.0, persistence_windows=2)
        if not alarms:
            missed += 1
            continue
        first_alarm = float(min(alarms))
        if first_alarm <= target_time:
            pre_entry_hits += 1
            early_times.append(target_time - first_alarm)
            grace_hits += 1
            grace_early_times.append(target_time - first_alarm)
        elif first_alarm <= target_time + float(late_grace_s):
            grace_hits += 1
            grace_early_times.append(target_time - first_alarm)
        else:
            missed += 1

    safe_df = df[df["is_danger_clip"] == 0]
    safe_minutes = 0.0
    safe_episodes = 0
    for _, group in safe_df.groupby("video_id"):
        if len(group):
            safe_minutes += float(group["time_s"].max()) / 60.0
            safe_episodes += len(alarm_episodes(group["time_s"], group[score_col], threshold, gap_s=1.0, persistence_windows=2))
    safe_minutes = max(1e-6, safe_minutes)
    return {
        "danger_clip_count": int(len(danger_clips)),
        "pre_entry_hit_rate": float(pre_entry_hits / max(1, len(danger_clips))),
        "late_grace_hit_rate": float(grace_hits / max(1, len(danger_clips))),
        "missed_or_too_late_clips": int(missed),
        "median_pre_entry_early_warning_s": float(np.median(early_times)) if early_times else np.nan,
        "median_grace_early_warning_s": float(np.median(grace_early_times)) if grace_early_times else np.nan,
        "safe_false_alarm_episodes": int(safe_episodes),
        "safe_false_alarms_per_min": float(safe_episodes / safe_minutes),
    }


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    run_dir = make_run_dir(args.run_name)
    ensemble_run = resolve(args.ensemble_run)
    operating_run = resolve(args.operating_run)
    prediction_files = dict(load_prediction_files(ensemble_run))
    operating = pd.read_csv(operating_run / "metrics" / "operating_points_by_seed.csv")
    operating = operating[(operating["variant"].isin(args.variants)) & (operating["policy"].isin(args.policies))].copy()
    write_json(
        run_dir / "config.json",
        {
            "ensemble_run": str(ensemble_run),
            "operating_run": str(operating_run),
            "variants": args.variants,
            "policies": args.policies,
            "target_shifts_s": args.target_shifts,
            "boundary_slacks_s": args.boundary_slacks,
            "late_graces_s": args.late_graces,
            "horizon_s": args.horizon,
        },
    )
    rows = []
    for seed, pred_path in prediction_files.items():
        pred = pd.read_csv(pred_path)
        test = pred[pred["split"] == "test"].copy()
        seed_ops = operating[operating["repeat_seed"].astype(int).eq(seed)]
        for _, op in seed_ops.iterrows():
            variant = op["variant"]
            if variant not in test:
                continue
            threshold = float(op["selected_threshold"])
            policy = op["policy"]
            for target_shift in args.target_shifts:
                for boundary_slack in args.boundary_slacks:
                    labels, _ = recompute_labels(test, args.horizon, target_shift, boundary_slack)
                    window_metrics = evaluate_windows(test, variant, labels)
                    for late_grace in args.late_graces:
                        op_metrics = evaluate_operating_threshold(test, variant, threshold, target_shift, late_grace)
                        rows.append(
                            {
                                "repeat_seed": seed,
                                "variant": variant,
                                "policy": policy,
                                "threshold": threshold,
                                "target_shift_s": float(target_shift),
                                "boundary_slack_s": float(boundary_slack),
                                "late_grace_s": float(late_grace),
                                **window_metrics,
                                **op_metrics,
                            }
                        )
    detail = pd.DataFrame(rows)
    detail.to_csv(run_dir / "metrics" / "label_timing_uncertainty_detail.csv", index=False)
    summary_rows = []
    group_cols = ["variant", "policy", "target_shift_s", "boundary_slack_s", "late_grace_s"]
    metric_cols = [
        "positive_windows",
        "window_ap",
        "window_roc_auc",
        "pre_entry_hit_rate",
        "late_grace_hit_rate",
        "missed_or_too_late_clips",
        "median_pre_entry_early_warning_s",
        "median_grace_early_warning_s",
        "safe_false_alarms_per_min",
    ]
    for keys, group in detail.groupby(group_cols):
        row = dict(zip(group_cols, keys))
        row["n_repeats"] = int(group["repeat_seed"].nunique())
        for col in metric_cols:
            row[f"{col}_mean"] = float(group[col].mean())
            row[f"{col}_std"] = float(group[col].std(ddof=0))
        summary_rows.append(row)
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(run_dir / "metrics" / "label_timing_uncertainty_summary.csv", index=False)

    base = summary[
        summary["target_shift_s"].eq(0.0)
        & summary["boundary_slack_s"].eq(0.0)
        & summary["late_grace_s"].eq(0.0)
        & summary["variant"].eq("mean_temporal_conv")
    ].copy()
    slack = summary[
        summary["target_shift_s"].eq(0.0)
        & summary["boundary_slack_s"].eq(0.5)
        & summary["late_grace_s"].eq(0.5)
        & summary["variant"].eq("mean_temporal_conv")
    ].copy()
    lines = ["# Label Timing Uncertainty Audit", ""]
    lines.append("This audit recomputes labels and hit metrics under shifted danger-event timestamps and temporal slack.")
    lines.append("")
    lines.append("## Mean Temporal-Conv Baseline vs 0.5s Slack")
    lines.append("")
    lines.append("| policy | label setting | AP | pre-entry hit | grace hit | median early s | FA/min |")
    lines.append("|---|---|---:|---:|---:|---:|---:|")
    for label_name, frame in [("strict", base), ("0.5s boundary + late grace", slack)]:
        for _, row in frame.sort_values("policy").iterrows():
            lines.append(
                f"| {row['policy']} | {label_name} | {row['window_ap_mean']:.3f} | "
                f"{row['pre_entry_hit_rate_mean']:.3f} | {row['late_grace_hit_rate_mean']:.3f} | "
                f"{row['median_pre_entry_early_warning_s_mean']:.3f} | {row['safe_false_alarms_per_min_mean']:.3f} |"
            )
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- If AP or hit rate changes strongly under +/- timestamp shifts, the event annotation is a major source of uncertainty.")
    lines.append("- Late-grace hit rate is useful because some human danger markers were approximate or marked at first visible concern rather than exact physical entry.")
    lines.append("- Safe false alarms do not change with label timing because safe clips have no event timestamp; they remain a threshold-policy issue.")
    lines.append("")
    lines.append("## Artifacts")
    lines.append("")
    lines.append(f"- Detail: `{run_dir / 'metrics' / 'label_timing_uncertainty_detail.csv'}`")
    lines.append(f"- Summary: `{run_dir / 'metrics' / 'label_timing_uncertainty_summary.csv'}`")
    summary_path = run_dir / "label_timing_uncertainty_summary.md"
    summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    append_report(run_dir, "Label Timing Uncertainty Audit", f"- Summary: `{summary_path}`")
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Evaluate final predictions under danger-event timestamp uncertainty.")
    parser.add_argument("--run-name", default="exp_038_label_timing_uncertainty")
    parser.add_argument("--ensemble-run", default="runs/exp_029_sequence_ensemble_stability")
    parser.add_argument("--operating-run", default="runs/exp_033_final_operating_points")
    parser.add_argument("--variants", nargs="+", default=["mean_temporal_conv", "mean_all8", "mean_tcn_only"])
    parser.add_argument("--policies", nargs="+", default=["max_hit_low_fa", "low_false_alarm", "balanced_window_f1"])
    parser.add_argument("--target-shifts", nargs="+", type=float, default=[-0.5, -0.25, 0.0, 0.25, 0.5])
    parser.add_argument("--boundary-slacks", nargs="+", type=float, default=[0.0, 0.25, 0.5])
    parser.add_argument("--late-graces", nargs="+", type=float, default=[0.0, 0.25, 0.5])
    parser.add_argument("--horizon", type=float, default=1.0)
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_038_label_timing_uncertainty_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["label_timing_uncertainty_audit.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
